# Building a Composite Index — HAI and XSTIR

**DS4DH Practice Pack · Module 08 — Index and Metric Design**

*Technique:* Constructing a number that does not exist in the data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/08b_composite_index.ipynb)

Data: `merged_dataset.csv`, `full_accessibility_index.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# This notebook reads the CSVs sitting next to it. In Colab you will be asked
# to upload them from the pack's data/ folder.
NEEDED = ['merged_dataset.csv', 'full_accessibility_index.csv']

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('merged_dataset.csv')
df_acc   = pd.read_csv('full_accessibility_index.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Nobody measures "housing accessibility". It is not a quantity in the world; it is
a number somebody decided to build, from components they chose, weighted the way
they thought right.

That is not a criticism — GDP, HDI and the consumer price index are all like this,
and all useful. It does mean the construction has to be visible, because every
step is a judgement someone could reasonably have made differently.

You will build two: **HAI** (Housing Accessibility Index) and **XSTIR** (an
extended burden measure), then check them against the shipped file.

In [ ]:
# One row per Census Subdivision.
#   • rows with no csd_code are CMA-level and Canada-level aggregates, not CSDs
#   • each CSD appears 3x (Immigrant / Non-immigrants / Total Immigrant Status)
# Keeping either would silently double- or triple-count places.
csd = df.dropna(subset=['csd_code'])
base = csd[(csd['immigrant_status'] == 'Total Immigrant Status')
           & (csd['cma'].isin(CITIES))].copy()

print(f'{len(df):>4} rows in the raw file')
print(f'{len(csd):>4} after dropping CMA/Canada aggregate rows')
print(f'{len(base):>4} CSDs in the four cities (one row each)')

In [ ]:
# HAI components: renter burden, renter income, and the renter-owner gap.
HAI_COLS = ['Renter', 'rent_income', 'renter_owner_gap']

hai_df = base.dropna(subset=HAI_COLS).copy()
print(f'{len(base)} CSDs -> {len(hai_df)} with all three HAI components')
print()
print(hai_df[HAI_COLS].describe().round(1).to_string())

## Step 1 — normalise, with polarity fixed

- `Renter` — high is worse → keep
- `rent_income` — high is better → invert
- `renter_owner_gap` — high is worse → keep

After this every component points the same way: **high = less accessible**.

In [ ]:
def minmax(s):
    return (s - s.min()) / (s.max() - s.min())

hai_df['n_burden'] = minmax(hai_df['Renter'])
hai_df['n_income'] = 1 - minmax(hai_df['rent_income'])
hai_df['n_gap'] = minmax(hai_df['renter_owner_gap'])

print(hai_df[['n_burden', 'n_income', 'n_gap']].describe().round(3).to_string())

In [ ]:
# Step 2 — combine. Equal weights, stated as a choice rather than assumed.
WEIGHTS = {'n_burden': 1 / 3, 'n_income': 1 / 3, 'n_gap': 1 / 3}

hai_df['HAI'] = sum(hai_df[c] * w for c, w in WEIGHTS.items())

print(f'HAI on {len(hai_df)} CSDs — 0 = most accessible, 1 = least')
print(hai_df['HAI'].describe().round(3).to_string())
print()
print('Least accessible:')
print(hai_df.nlargest(6, 'HAI')[
    ['geography_name', 'cma', 'Renter', 'rent_income', 'HAI']].to_string(index=False))

In [ ]:
# Cross-check against the shipped accessibility index.
merged = hai_df.merge(
    df_acc[['csd_code', 'accessibility_index']], on='csd_code', how='inner')

print(f'{len(merged)} CSDs present in both')
print(f'correlation between our HAI and the shipped index: '
      f'{merged["HAI"].corr(merged["accessibility_index"]):.3f}')
print()
print('The shipped index runs the other way — high = MORE accessible — so a')
print('strong negative correlation is the expected agreement.')

## XSTIR — extending a metric rather than accepting it

STIR asks what share of income goes to housing. It says nothing about what is
left. A household spending 30% of $200,000 and one spending 30% of $30,000 are
not in the same situation, and STIR scores them identically.

XSTIR adds residual income to the picture: burden, adjusted for what remains.

In [ ]:
RESIDUAL_FLOOR = 25000   # a stated assumption, not a fact

x = hai_df.copy()
x['residual'] = x['rent_income'] * (1 - x['Renter'] / 100)
x['shortfall'] = (RESIDUAL_FLOOR - x['residual']).clip(lower=0)
x['XSTIR'] = x['Renter'] + 100 * x['shortfall'] / x['rent_income']

print(f'{"":<26}{"STIR":>10}{"XSTIR":>10}{"diff":>9}')
print('-' * 55)
for _, r in x.nlargest(8, 'XSTIR').iterrows():
    print(f'{r["geography_name"][:25]:<26}{r["Renter"]:>10.1f}'
          f'{r["XSTIR"]:>10.1f}{r["XSTIR"] - r["Renter"]:>+9.1f}')
print()
print(f'{(x["shortfall"] > 0).sum()} of {len(x)} CSDs fall below the residual floor.')

In [ ]:
fig, ax = plt.subplots()
ax.scatter(x['Renter'], x['XSTIR'], s=24, alpha=0.6)
lim = [x['Renter'].min(), x['XSTIR'].max()]
ax.plot(lim, lim, color='#E8663D', ls='--', label='XSTIR = STIR (no adjustment)')
ax.set_xlabel('STIR (%)')
ax.set_ylabel('XSTIR (%)')
ax.set_title(f'Places above the line have residual income below ${RESIDUAL_FLOOR:,}')
ax.legend()
plt.tight_layout()
plt.show()

### 🔧 Your turn 1

Change `RESIDUAL_FLOOR` to 15000, then 40000.

How many CSDs are flagged at each? The floor is the entire policy content of
XSTIR — where would you get a defensible value for it, rather than choosing one?

### 🔧 Your turn 2

Rank the CSDs by STIR and by XSTIR, and count how many places move more than 10
positions.

If the two rankings largely agree, is XSTIR worth the extra assumption?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** The count rises steeply with the floor — it is the single most
consequential number in the metric. A defensible value would come from an
external standard rather than from you: a market-basket measure of basic needs,
Statistics Canada's low-income measure, or a provincial income-assistance
threshold, adjusted for household size and region. Choosing a round number
yourself makes the whole index your opinion.

**Your turn 2.** The rankings agree for most places and diverge sharply for
low-income CSDs, which is exactly the population XSTIR was built to surface. That
is the argument for it: a metric is worth an extra assumption when it changes the
answer *for the group you care about*, even if it leaves the overall ranking
mostly intact. If it moved nobody, it would be complexity for its own sake.

</details>

## Where this stops

You have two constructed metrics and a rank ordering of municipalities. Every
number in them rests on the equal weights chosen in step 2 — which have not yet
been examined at all. That is the next notebook.